[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/junlinghu/book-agents/blob/main/tutorial/1-using-tool.ipynb)

## Colab setup

On Google Colab, use the setup in `tutorial/common/colab.py`. A fresh Colab runtime does not have this repository, so clone the repo into the session first. Then, before the lesson cells, run `setup_colab` from that module (import it from `tutorial.common.colab` and call it, or open the file and run `setup_colab` from there). The helper installs the tutorial packages, checks the repo out under `/content/book-agents` when it is still missing, and copies a Colab secret named `OPENAI_API_KEY`. Local Jupyter and VS Code can skip it when you already have the repo.

An API key is required. Set `OPENAI_API_KEY` in Colab secrets (the key icon, secret name `OPENAI_API_KEY`) before you run `setup_colab`. Locally, put the same name in the repository-root `.env`. A missing key stops the notebook. The key is not printed.

Edits you make in Colab stay in that session. They do not push to GitHub.

# 1. Tool calling

One tool, two model calls. The model asks for `get_shop_fact`. This notebook reads the Returns section of `docs/policy.md` and sends that text back. The second call writes the answer from that text.

You need `OPENAI_API_KEY` in the repository-root `.env`, or in a Colab secret of the same name.


## Setup

Find the repo so `import tutorial` works, then load the key. `chat` calls the OpenAI Chat Completions API. The key is not printed.


In [ ]:
import sys
from pathlib import Path


def find_root():
    here = Path.cwd().resolve()
    for folder in [here, *here.parents]:
        if (folder / "tutorial" / "common" / "client.py").is_file():
            return folder
    raise RuntimeError("Open this notebook inside the book-agents repository.")


ROOT = find_root()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from tutorial.runtime import chat, require_key

_, model = require_key()
print("model:", model)


## Test chat

One Chat Completions call, with no tools. This checks the key and the model before the tool run.


In [ ]:
def test_chat(question):
    """Ask one question with no tools, and print the answer."""
    turn = chat([{"role": "user", "content": question}], tools=[])
    print("ANSWER:", turn["content"])


test_chat("Reply with exactly: ok")


## One tool

`get_shop_fact` opens a shop document and returns one section. The model only sees that string. `handlers` maps the name in the schema to the Python function.


In [ ]:
def get_shop_fact(args):
    """Open one shop document and return the section for this topic."""
    topic = str(args.get("topic", "")).strip().lower()
    pages = {
        "returns": ("policy.md", "Returns"),
        "shipping": ("policy.md", "Shipping"),
        "hours": ("faq.md", "Where and when"),
        "allergens": ("faq.md", "Allergens"),
    }
    if topic not in pages:
        return "topic must be returns, shipping, hours, or allergens."
    filename, heading = pages[topic]
    lines = (ROOT / "tutorial" / "docs" / filename).read_text().splitlines()
    start = lines.index("## " + heading) + 1
    body = []
    for line in lines[start:]:
        if line.startswith("## "):
            break
        body.append(line)
    text = "\n".join(body).strip()
    return "TOPIC: " + topic + "\nSOURCE: docs/" + filename + "\n\n" + text


tools = [{
    "type": "function",
    "function": {
        "name": "get_shop_fact",
        "description": "Look up one shop rule. topic is returns, shipping, hours, or allergens.",
        "parameters": {
            "type": "object",
            "properties": {
                "topic": {"type": "string", "description": "returns, shipping, hours, or allergens"},
            },
            "required": ["topic"],
        },
    },
}]

handlers = {
    "get_shop_fact": get_shop_fact,
}


## 1. Ask

Send the question and the tool list. This first call should request `get_shop_fact`. It does not open the file.


In [ ]:
question = "Can a customer return an opened bag of house coffee?"
messages = [
    {
        "role": "system",
        "content": (
            "You are the counter concierge for Hearth Lane Café. "
            "Use get_shop_fact before you state a return, shipping, hours, or allergen rule. "
            "Answer from that tool result."
        ),
    },
    {"role": "user", "content": question},
]
print("QUESTION:", question)

reply = chat(messages, tools)
print("tool_calls:", [call["name"] for call in reply["tool_calls"]])


## 2. Run the tool

The model returns a tool name as a string. Look that name up in `handlers` and call the function with `call["arguments"]`.


In [ ]:
call = reply["tool_calls"][0]
name = call["name"]
messages.append({
    "role": "assistant",
    "content": reply["content"] or "",
    "tool_calls": [{
        "id": call["id"],
        "type": "function",
        "function": {"name": name, "arguments": call["raw"]},
    }],
})
result = handlers[name](call["arguments"])
print(name, result)
messages.append({"role": "tool", "tool_call_id": call["id"], "content": result})


## 3. Answer

Call the model again. `reply` is now the second turn. It should answer from the tool result.


In [ ]:
reply = chat(messages, tools)
print("ANSWER:", reply["content"])


## Chat

One loop. It reads a line, then calls the model. If the model requests a tool, the loop runs it and calls the model again without asking you. If the model replies with text, the notebook prints it and asks for your next line. A text reply does not end the chat. Only `bye` does.


In [ ]:
print("Ask a question. Type bye to end.")

messages = [
    {
        "role": "system",
        "content": (
            "You are the counter concierge for Hearth Lane Café. "
            "Use get_shop_fact only for a return, shipping, hours, or allergen rule. "
            "For any other question, answer directly and do not call a tool."
        ),
    },
]
# After a tool, the next pass calls the model again. Otherwise it asks you.
need_input = True

while True:
    if need_input:
        question = input("You: ").strip()
        if question.lower() == "bye":
            print("Bye.")
            break
        if question == "":
            continue
        messages.append({"role": "user", "content": question})

    reply = chat(messages, tools)
    if reply["tool_calls"]:
        pending = []
        for call in reply["tool_calls"]:
            pending.append({
                "id": call["id"],
                "type": "function",
                "function": {"name": call["name"], "arguments": call["raw"]},
            })
        messages.append({
            "role": "assistant",
            "content": reply["content"] or "",
            "tool_calls": pending,
        })
        for call in reply["tool_calls"]:
            name = call["name"]
            result = handlers[name](call["arguments"])
            print(name, result)
            messages.append({
                "role": "tool",
                "tool_call_id": call["id"],
                "content": result,
            })
        need_input = False
    else:
        messages.append({"role": "assistant", "content": reply["content"] or ""})
        print("Café:", reply["content"])
        print()
        need_input = True
